# 🌱 Greedy Algorithms & Amortized Analysis -- Runnable Notebook

Companion to [`README.md`](README.md).

Interval scheduling, interval partitioning (greedy + heap), a greedy failure case, and amortized dynamic-array doubling.

## 1. When greedy FAILS -- coin change with arbitrary denominations

In [ ]:
def greedy_coin_change(coins, target):
    """Greedily take the largest coin that fits. NOT always optimal."""
    coins = sorted(coins, reverse=True)
    count, remaining = 0, target
    used = []
    for c in coins:
        while remaining >= c:
            remaining -= c
            used.append(c)
            count += 1
    return used if remaining == 0 else None

# With US-like denominations, greedy happens to be optimal.
us_result = greedy_coin_change([25, 10, 5, 1], 30)
print("US coins for 30:", us_result)
assert len(us_result) == 2   # 25 + 5

# With {1, 3, 4} and target 6, greedy is WRONG: it picks 4+1+1 (3 coins) instead of 3+3 (2 coins).
bad_result = greedy_coin_change([1, 3, 4], 6)
print("greedy coins for 6 with {1,3,4}:", bad_result, "-- suboptimal! optimal is [3, 3] (2 coins)")
assert len(bad_result) == 3          # greedy's answer: worse than the optimal 2-coin answer
assert sum(bad_result) == 6           # still a VALID answer, just not optimal count

## 2. Interval scheduling -- max non-overlapping intervals (sort by END time)

In [ ]:
def max_non_overlapping(intervals):
    intervals = sorted(intervals, key=lambda iv: iv[1])   # sort by END time
    count, last_end = 0, float("-inf")
    for start, end in intervals:
        if start >= last_end:
            count += 1
            last_end = end
    return count

meetings = [(1, 3), (2, 4), (3, 5), (0, 6), (5, 7), (8, 9)]
result = max_non_overlapping(meetings)
print("max non-overlapping meetings:", result)
assert result == 4     # (1,3), (3,5), (5,7), (8,9) -- four non-overlapping meetings fit

# Sanity check: sorting by START time instead can give a worse answer.
def max_non_overlapping_wrong(intervals):
    intervals = sorted(intervals, key=lambda iv: iv[0])   # sort by START time -- the WRONG rule
    count, last_end = 0, float("-inf")
    for start, end in intervals:
        if start >= last_end:
            count += 1
            last_end = end
    return count

wrong_result = max_non_overlapping_wrong(meetings)
print("sorting by start time instead gives:", wrong_result)
assert wrong_result <= result    # start-time sorting is never BETTER than end-time sorting

## 3. Interval partitioning -- min resources needed (sort by START + heap)

In [ ]:
import heapq

def min_resources(intervals):
    intervals = sorted(intervals, key=lambda iv: iv[0])
    heap = []
    for start, end in intervals:
        if heap and heap[0] <= start:
            heapq.heappop(heap)
        heapq.heappush(heap, end)
    return len(heap)

def max_overlap(intervals):
    """Independent check via sweep-line: min resources should equal the peak overlap."""
    events = []
    for s, e in intervals:
        events.append((s, 1)); events.append((e, -1))
    events.sort(key=lambda ev: (ev[0], ev[1]))   # ends before starts at same timestamp -> touching doesn't overlap
    active = best = 0
    for _, delta in events:
        active += delta
        best = max(best, active)
    return best

example = [(0, 10), (5, 15), (10, 20), (20, 30)]
resources = min_resources(example)
overlap = max_overlap(example)
print("min resources needed:", resources, "| max simultaneous overlap:", overlap)
assert resources == overlap == 2   # [0,10) & [5,15) overlap; [10,20)&[20,30) touch but don't overlap

triple_overlap = [(0, 100), (0, 100), (0, 100)]
assert min_resources(triple_overlap) == max_overlap(triple_overlap) == 3

## 4. Amortized analysis -- dynamic array doubling

In [ ]:
class DoublingArray:
    def __init__(self):
        self.capacity = 1
        self.size = 0
        self.data = [None] * self.capacity
        self.resize_events = []           # capacities at which a resize happened
        self.total_copy_work = 0           # total elements ever copied during resizes

    def append(self, x):
        if self.size == self.capacity:
            self.capacity *= 2
            new_data = [None] * self.capacity
            for i in range(self.size):
                new_data[i] = self.data[i]
                self.total_copy_work += 1
            self.data = new_data
            self.resize_events.append(self.capacity)
        self.data[self.size] = x
        self.size += 1

arr = DoublingArray()
N = 1000
for i in range(N):
    arr.append(i)

print(f"appended {N} items")
print("resize events (capacities):", arr.resize_events)
print("total elements copied across ALL resizes:", arr.total_copy_work)

# The key amortized-analysis claim: total copy work across N appends is O(N), not O(N^2).
# A naive "resize by +1 every time" strategy would copy 0+1+2+...+N = O(N^2) elements total.
assert arr.total_copy_work < 2 * N          # doubling keeps total copying under 2N
naive_total_copy_work = sum(range(N))        # what a "grow by 1 every append" strategy would cost
print("compare to a naive +1-at-a-time growth strategy's total copy work:", naive_total_copy_work)
assert arr.total_copy_work < naive_total_copy_work   # doubling is dramatically cheaper in total

## ✅ Recap
- **Exchange argument**: proves a greedy choice is safe by showing it can always be swapped into an optimal solution without making it worse.
- Greedy does **not** always work -- coin change with arbitrary denominations is the classic counterexample; always verify the exchange argument holds.
- **Interval scheduling** (max non-overlapping): sort by **end time**.
- **Interval partitioning** (min resources): sort by **start time** + a **min-heap** of availability times; the answer equals the **max simultaneous overlap**.
- **Amortized analysis**: total cost across a *sequence* of operations is bounded, even when individual operations vary wildly -- dynamic array doubling is the textbook example (O(1) amortized per append, despite occasional O(n) resizes).

This closes the Algorithmic Foundations set: [`16_Hash_Tables`](../16_Hash_Tables/README.md), [`17_Sorting_Algorithms`](../17_Sorting_Algorithms/README.md), [`18_Binary_Search`](../18_Binary_Search/README.md), [`19_Two_Pointers_Sliding_Window`](../19_Two_Pointers_Sliding_Window/README.md), and this one -- together with the Trees/Graphs/Heaps/Tries tracks, these cover every technique used across `Atlassian_Prep/` (see `Atlassian_Prep/DSA_Used.md`).